In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, expr, regexp_replace

spark = SparkSession.builder.getOrCreate()

df = spark.table("main.bronze.qualite_eau_region")

df = df.drop(
    "libelle_parametre_web",
    "code_installation_amont",
    "nom_installation_amont"
)

df = df.withColumn("date_prelevement", to_timestamp(col("date_prelevement")))

code_cols = [c for c in df.columns if c.startswith("code")]
for c in code_cols:
    df = df.withColumn(c, col(c).cast("string"))

df = df.withColumn("code_prelevement", col("code_prelevement").cast("string"))
df = df.withColumn("reference_analyse", col("reference_analyse").cast("string"))

df = df.withColumn("resultat_alphanumerique", col("resultat_alphanumerique").cast("string"))

df = df.withColumn(
    "resultat_numerique",
    expr("""
        try_cast(
            regexp_replace(
                regexp_replace(resultat_alphanumerique, ',', '.'),
                '<',
                ''
            ) as double
        )
    """)
)

df = df.withColumn("reseau_code", col("reseaux")[0]["code"].cast("string"))
df = df.withColumn("reseau_nom", col("reseaux")[0]["nom"].cast("string"))
df = df.drop("reseaux")

df = df.withColumn("parametre", col("libelle_parametre"))
df = df.withColumn("unite", col("libelle_unite"))

df = df.withColumn("limite_qualite_parametre", col("limite_qualite_parametre"))
df = df.withColumn("reference_qualite_parametre", col("reference_qualite_parametre"))

df = df.withColumn("commune", col("nom_commune"))
df = df.withColumn("departement", col("nom_departement"))

df = df.withColumn(
    "conformite",
    col("conclusion_conformite_prelevement").cast("string")
)

silver_df = df.select(
    "code_prelevement",
    "date_prelevement",

    "departement",
    "commune",

    "parametre",
    "libelle_parametre",

    "resultat_alphanumerique",
    "resultat_numerique",

    "unite",
    "limite_qualite_parametre",
    "reference_qualite_parametre",

    "reseau_code",
    "reseau_nom",

    "conformite_limites_bact_prelevement",
    "conformite_limites_pc_prelevement",
    "conformite_references_bact_prelevement",
    "conformite_references_pc_prelevement",

    "code_departement",
    "code_commune",
    "reference_analyse"
)

silver_df.write.mode("overwrite").saveAsTable("main.silver.qualite_eau_region")


In [ ]:
%sql
--qualité globale de l'eau
SELECT
  date_prelevement,
  commune,
  parametre,
  resultat_numerique,

  try_cast(
    regexp_replace(
      regexp_replace(limite_qualite_parametre, ',', '.'),
      '[^0-9.]',
      ''
    ) as double
  ) AS limite_numerique,

  CASE
    WHEN resultat_numerique >
         try_cast(
           regexp_replace(
             regexp_replace(limite_qualite_parametre, ',', '.'),
             '[^0-9.]',
             ''
           ) as double
         )
    THEN 1 ELSE 0
  END AS depassement

FROM main.silver.qualite_eau_region
WHERE resultat_numerique IS NOT NULL

In [ ]:
%sql
-- ÉVOLUTION DU pH (TA BASE — TRÈS BIEN
SELECT
  reseau_nom,
  COUNT(*) AS nb_mesures,
  SUM(CASE WHEN upper(conformite_limites_pc_prelevement) LIKE '%NC%' THEN 1 ELSE 0 END) AS nc_pc,
  SUM(CASE WHEN upper(conformite_limites_bact_prelevement) LIKE '%NC%' THEN 1 ELSE 0 END) AS nc_bact
FROM main.silver.qualite_eau_region
GROUP BY reseau_nom
ORDER BY nc_pc DESC

In [ ]:
%sql
--parametres les plus critiques
SELECT
  date_trunc('day', date_prelevement) AS jour,
  AVG(resultat_numerique) AS ph_moyen
FROM main.silver.qualite_eau_region
WHERE upper(trim(parametre)) LIKE '%PH%'
GROUP BY jour
ORDER BY jour